# 05 · Configurar la herramienta

## Propósito

En el `04` viste los errores. Acá los corregimos.

El recorrido es uno solo: definir reglas propias, meterlas adentro del pipeline
para que corran solas, agregarle al documento atributos que spaCy no trae, y
guardar todo eso en disco para poder volver a usarlo y compartirlo.

Al final vas a tener una herramienta configurada por vos, no el modelo que viene
de fábrica.

## Qué consume

- `datos/corpus.jsonl`
- `datos/personajes.txt`

## Qué produce

- `pipeline_propio/`

## ◈ Microglosario

| Término | Qué es |
|---|---|
| **patrón** | La descripción de lo que buscás, escrita como lista de condiciones sobre tokens |
| **`Matcher`** | Busca patrones usando atributos lingüísticos: lema, POS, morfología |
| **`PhraseMatcher`** | Busca una lista de términos literales. Más rápido, menos flexible |
| **`EntityRuler`** | Un componente que mete tus reglas **adentro** del pipeline, para que sus resultados salgan en `doc.ents` |
| **componente** | Una función que recibe un `Doc`, lo modifica y lo devuelve. Nada más que eso |

## Celda de reanudación

In [ ]:
# Recarga automática si editamos archivos externos (.py)
%load_ext autoreload
%autoreload 2

# spacy: la librería de análisis lingüístico.
import spacy

# displacy: el visualizador que viene adentro de spaCy.
from spacy import displacy

# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# es_core_news_lg: modelo en español (es), noticias (news), grande (lg)
nlp = spacy.load("es_core_news_lg")

# lines=True: lee formato .jsonl (un JSON por renglón)
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

# pipe_names: lista de componentes activos en el pipeline
print("Componentes activos:", nlp.pipe_names)

## 1. `Matcher`: buscar por patrones

`Matcher` busca secuencias de tokens que cumplan condiciones. La diferencia con
una expresión regular es fundamental: **una regex trabaja sobre caracteres, el
`Matcher` trabaja sobre tokens anotados.** Podés pedir "un verbo seguido de un
sustantivo" sin escribir qué palabras son.

Empecemos por una locución fija.

In [ ]:
from spacy.matcher import Matcher

# Matcher necesita el vocabulario compartido del modelo
buscador = Matcher(nlp.vocab)

# LOWER: compara en minúsculas (ignora mayúsculas del texto)
patron_a_la_gorra = [
    {"LOWER": "a"},
    {"LOWER": "la"},
    {"LOWER": "gorra"}
]

# Va entre corchetes [patron] porque una regla puede tener varias alternativas
buscador.add("LOCUCION", [patron_a_la_gorra])

doc = nlp("La revista se vendía a la gorra y funcionó a la gorra durante años.")
# Al buscador se lo usa como una función: le pasamos el documento y devuelve lo que encontró.
# as_spans=True: que devuelva cada coincidencia como Span, igual que las oraciones del 01
coincidencias = buscador(doc, as_spans=True)

cantidad_de_coincidencias = len(coincidencias)
print(f"Coincidencias encontradas: {cantidad_de_coincidencias}")
# start y end: dónde empieza y dónde termina el tramo, contado en tokens (no en letras)
for tramo in coincidencias:
    print(f"• '{tramo.text}' (tokens {tramo.start} a {tramo.end})")

### Patrones con atributos lingüísticos

Acá está la potencia real. En vez de listar palabras, describís categorías.

In [ ]:
buscador_gramatical = Matcher(nlp.vocab)

# POS: Part of Speech (categoría gramatical). NOUN: sustantivo, ADJ: adjetivo
patron_sustantivo_adjetivo = [
    {"POS": "NOUN"},
    {"POS": "ADJ"}
]

buscador_gramatical.add("SUSTANTIVO_ADJETIVO", [patron_sustantivo_adjetivo])

# iloc[101]: fila 101 del DataFrame
texto_ejemplo = corpus["texto"].iloc[101]
doc = nlp(texto_ejemplo)

print(f'Texto: "{texto_ejemplo}"')
print()

coincidencias = buscador_gramatical(doc, as_spans=True)
for tramo in coincidencias:
    print("•", tramo.text)

### Operadores

Cada token del patrón acepta un operador `OP`:

| operador | significa |
|---|---|
| `?` | cero o una vez |
| `*` | cero o más veces |
| `+` | una o más veces |
| `!` | que NO sea así |

Con eso se capturan estructuras variables.

In [ ]:
buscador_flexible = Matcher(nlp.vocab)

# LEMMA: forma de diccionario ("dar"). OP "*": cero o más palabras intermedias
patron_dar_algo = [
    {"LEMMA": "dar"},
    {"OP": "*"},
    {"POS": "NOUN"}
]

buscador_flexible.add("DAR_ALGO", [patron_dar_algo])

pruebas = [
    "Le dio un beso.",
    "Dio una larga y aburrida charla.",
    "Nos dieron la razón."
]

for frase in pruebas:
    doc = nlp(frase)
    print()
    print(f"Frase: '{frase}'")
    coincidencias = buscador_flexible(doc, as_spans=True)
    for tramo in coincidencias:
        print(f"  → '{tramo.text}'")

> **Ojo con `*`.** Devuelve **todas** las coincidencias posibles, incluidas las
> superpuestas: si hay dos sustantivos después del verbo, vas a recibir dos
> resultados anidados. Para quedarte con el más largo de cada grupo, spaCy trae
> `spacy.util.filter_spans()`.

## 2. `PhraseMatcher`: buscar muchos términos

El `Matcher` sirve para describir estructuras. Cuando lo que tenés es **una lista
larga de nombres concretos**, el `PhraseMatcher` es mucho más rápido y mucho más
simple de escribir: le pasás los términos como texto y él los tokeniza igual que
el modelo.

Nuestra lista está en `datos/personajes.txt`. **Abrila antes de seguir.**

In [ ]:
entradas = []

# with open(...): abre el archivo y lo cierra solo al terminar el bloque.
# encoding="utf-8": para que lea bien las tildes y la ñ
with open("datos/personajes.txt", encoding="utf-8") as archivo:
    for linea in archivo:
        linea = linea.strip()  # quita espacios y saltos de línea

        # Paso 1: Si no tiene la barra "|", no nos sirve
        if "|" in linea:
            # Paso 2: Si empieza con "#", es un comentario y no lo queremos
            if not linea.startswith("#"):
                # split("|"): corta la línea en dos partes, una a cada lado de la barra
                categoria, texto = linea.split("|")
                # Guardamos el par (categoría, texto)
                entradas.append((categoria, texto))

cantidad_de_entradas = len(entradas)
print(f"Entradas leídas: {cantidad_de_entradas}")
# [:6]: las primeras seis entradas
for categoria, texto in entradas[:6]:
    print(f"• {categoria} -> {texto}")

In [ ]:
from spacy.matcher import PhraseMatcher

# PhraseMatcher: buscador ultra-rápido para listas de nombres o frases exactas
buscador_nombres = PhraseMatcher(nlp.vocab)

# Agrupamos los textos en un diccionario: {"PERSONAJE": [doc1, doc2], ...}
por_categoria = {}
for categoria, texto in entradas:
    # Si la categoría todavía no está en el diccionario, creamos su lista vacía
    if categoria not in por_categoria:
        por_categoria[categoria] = []

    # make_doc(): solo corta en tokens (palabras), sin hacer análisis lingüístico pesado
    doc_termino = nlp.make_doc(texto)
    por_categoria[categoria].append(doc_termino)

# Registramos cada categoría con su lista de términos en el buscador
# items(): cada clave (la categoría) con su valor (la lista de términos)
for categoria, terminos_de_la_categoria in por_categoria.items():
    buscador_nombres.add(categoria, terminos_de_la_categoria)

# .keys(): nombres de las categorías que quedaron registradas. list(): los pasa a una lista
claves_de_categorias = por_categoria.keys()
categorias_cargadas = list(claves_de_categorias)
print("Categorías cargadas:", categorias_cargadas)

`nlp.make_doc()` en vez de `nlp()` no es un detalle: solo tokeniza, sin correr
el resto del pipeline. Para buscar términos literales alcanza con los tokens, así
que el resto del análisis sería trabajo tirado.

Probemos el buscador sobre una frase.

In [ ]:
texto_prueba = "Mirta Bertotti vivía en Mercedes, pero Chiri se fue a Barcelona con la Revista Orsai."
doc = nlp.make_doc(texto_prueba)

# as_spans=True: cada coincidencia vuelve como Span, con su categoría en label_
coincidencias = buscador_nombres(doc, as_spans=True)

print(f"Texto: '{texto_prueba}'")
print()
for tramo in coincidencias:
    # label_: la categoría con la que registramos el término en el buscador
    print(f"• {tramo.text} -> {tramo.label_}")

## 3. `EntityRuler`: meter las reglas adentro del pipeline

Los dos buscadores anteriores hay que llamarlos a mano cada vez. El
`EntityRuler` es un componente del pipeline: se agrega una vez y corre solo,
como el `ner`.

Y hay algo más importante: **puede corregir al modelo estadístico.**

In [ ]:
# Cargamos una copia limpia del modelo para no arrastrar pruebas previas
nlp_mejorado = spacy.load("es_core_news_lg")

# overwrite_ents=True: si la regla choca con el modelo estadístico, gana nuestra regla
configuracion_de_reglas = {"overwrite_ents": True}

# add_pipe: agrega un componente nuevo a la tubería, y lo devuelve para que le carguemos reglas
# before="ner": lo coloca ANTES del reconocedor estadístico (ner)
reglas = nlp_mejorado.add_pipe("entity_ruler", config=configuracion_de_reglas, before="ner")

print("Pipeline con entity_ruler:")
print(nlp_mejorado.pipe_names)

In [ ]:
# EntityRuler pide diccionarios con la forma: {"label": "CATEGORIA", "pattern": "TEXTO"}
patrones = []
for categoria, texto in entradas:
    # label: etiqueta (ej: PERSONAJE). pattern: el texto exacto a buscar
    patron = {"label": categoria, "pattern": texto}
    patrones.append(patron)

# add_patterns(): carga toda la lista de reglas juntas
reglas.add_patterns(patrones)

cantidad_de_patrones = len(patrones)
print(f"Patrones cargados en el EntityRuler: {cantidad_de_patrones}")

### Antes y después

Comparemos el modelo de fábrica con el nuestro sobre la misma frase.

In [ ]:
frase = "Mirta Bertotti vivía en Mercedes y Chiri fundó la Revista Orsai en Barcelona."

# Comparamos el modelo original vs el modelo con nuestras reglas
doc_original = nlp(frase)
doc_mejorado = nlp_mejorado(frase)

print("--- 1. Modelo de fábrica ---")
# .ents: lista de entidades detectadas en el texto
# .label_: nombre de la etiqueta (LOC, PER, ORG, etc.)
for ent in doc_original.ents:
    print(f"• {ent.text} -> {ent.label_}")

print()
print("--- 2. Con nuestras reglas ---")
for ent in doc_mejorado.ents:
    print(f"• {ent.text} -> {ent.label_}")

In [ ]:
# displacy.render: genera el gráfico en el cuaderno
# style="ent": resalta entidades con etiquetas de colores
# jupyter=True: que lo dibuje acá adentro del cuaderno
displacy.render(doc_mejorado, style="ent", jupyter=True)

> **Lo que acabás de hacer no es un ajuste técnico.**
>
> Decidiste que *Revista Orsai* es un PROYECTO y no un lugar, y que *Mercedes*
> es un LUGAR. Esas son afirmaciones sobre el corpus, no configuraciones.
>
> Y `Mercedes` es el caso incómodo: es la ciudad natal del autor **y** un nombre
> de mujer. Tu regla la marca siempre como lugar, incluso cuando el texto habla
> de una persona. Probalo con `nlp_mejorado("Mercedes Carreras actuó en la obra.")`
> y mirá qué sale.
>
> No hay solución correcta. Hay una decisión, y hay que escribirla.

In [ ]:
# Probamos el caso límite: "Mercedes" como nombre de persona y no como ciudad
doc_ambiguo = nlp_mejorado("Mercedes Carreras actuó en la obra.")

print("Entidades reconocidas en el caso ambiguo:")
for ent in doc_ambiguo.ents:
    print(f"• {ent.text} -> {ent.label_}")

## 4. Un componente propio

El `EntityRuler` viene hecho. Cuando lo que necesitás no existe, se escribe.

Un componente de spaCy es una función que recibe un `Doc`, le hace algo y lo
devuelve. Vamos a escribir uno que corrija los lemas del voseo, usando el
diccionario del cuaderno `04`.

In [ ]:
# Language: clase para registrar componentes nuevos en spaCy
from spacy.language import Language

# Diccionario de equivalencias: palabra conjugada -> lema en infinitivo
CORRECCIONES_VOSEO = {
    "tenés": "tener",
    "sabés": "saber",
    "sos": "ser",
    "querés": "querer",
    "podés": "poder",
    "venís": "venir",
    "hacés": "hacer",
    "decís": "decir",
    "andá": "andar",
    "mirá": "mirar",
    "vení": "venir"
}

Un componente de spaCy es una función que recibe un `Doc`, lo modifica y lo devuelve.
Nada más que eso.

El decorador es lo único nuevo: le pone un nombre al componente dentro de spaCy, para
poder agregarlo al pipeline después.

In [ ]:
# @Language.component: registra la función como un componente oficial de spaCy
@Language.component("corrector_voseo_demo")
def corrector_voseo(doc):
    """Reemplaza el lema de las formas del voseo por su infinitivo."""
    for token in doc:
        # lower(): en minúsculas, para que "Tenés" también coincida con "tenés"
        palabra = token.text.lower()
        if palabra in CORRECCIONES_VOSEO:
            # lemma_ (con guion bajo): lema en texto legible. Modificarlo cambia el documento
            token.lemma_ = CORRECCIONES_VOSEO[palabra]
    return doc

Registrado, pero todavía no está en ningún pipeline. Falta agregarlo.

In [ ]:
# last=True: se ejecuta al final de todo (para pisar lo que hizo el lematizador estándar)
nlp_mejorado.add_pipe("corrector_voseo_demo", last=True)

print("Pipeline actualizado:", nlp_mejorado.pipe_names)

In [ ]:
doc_prueba = nlp_mejorado("Vos tenés razón, sabés que sos así y hacés lo que querés.")

filas = []
for token in doc_prueba:
    # is_punct: True si el token es un signo de puntuación (lo salteamos)
    if not token.is_punct:
        fila = {"token": token.text, "lema": token.lemma_}
        filas.append(fila)

# pd.DataFrame: arma una tabla visual limpia para ver los resultados
pd.DataFrame(filas)

**El orden importa.** Si hubiéramos puesto el corrector antes del
`lemmatizer`, el lemmatizer habría pisado nuestras correcciones y no se notaría
nada. Con `last=True` corre al final y gana.

Probalo al revés si querés verlo: `nlp.add_pipe("corrector_voseo_demo", before="lemmatizer")`.

## 5. Atributos propios

Un `Doc` trae los atributos que spaCy define. Podés agregarle los tuyos con
`set_extension`, y quedan disponibles bajo el prefijo `._`.

Vamos a agregarle a cada documento la lista de personajes que aparecen en él.

Fijate que el componente **no busca nada por su cuenta**: lee las entidades que
ya dejó el `EntityRuler`. Los componentes se encadenan, y cada uno aprovecha lo
que hizo el anterior. Por eso este tiene que ir después.

In [ ]:
# Doc: la clase de los documentos de spaCy
from spacy.tokens import Doc

# Doc.set_extension: crea una propiedad nueva en los documentos
# force=True: permite volver a correr la celda sin error si ya existía
Doc.set_extension("personajes", default=None, force=True)

@Language.component("detector_personajes_demo")
def detector_personajes(doc):
    """Guarda en doc._.personajes los nombres marcados como PERSONAJE."""
    encontrados = []
    for ent in doc.ents:

        # Solo nos interesan las entidades marcadas como PERSONAJE...
        if ent.label_ == "PERSONAJE":

            # ...que todavía no hayamos agregado.
            if ent.text not in encontrados:
                encontrados.append(ent.text)

    # doc._.personajes: el punto y guion bajo (._.) es donde spaCy guarda datos personalizados
    doc._.personajes = encontrados
    return doc

nlp_mejorado.add_pipe("detector_personajes_demo", last=True)
print("Pipeline:", nlp_mejorado.pipe_names)

In [ ]:
doc = nlp_mejorado("Mirta Bertotti le contó a Chiri que Nina había llegado.")

# Leemos directamente nuestro atributo personalizado
print("Personajes detectados en la oración:")
print(doc._.personajes)

Ahora ese atributo viaja con el documento. Podemos recorrer el corpus y
armar una tabla sin volver a buscar nada.

In [ ]:
# Counter: para contar.
from collections import Counter

# Tomamos una muestra de 600 párrafos
primeros = corpus.head(600)
textos_muestra = list(primeros["texto"])

# Counter(): contador automático de apariciones
conteo = Counter()

# nlp.pipe(): procesa muchos textos en lotes (batch_size=50: de a 50 textos)
for doc in nlp_mejorado.pipe(textos_muestra, batch_size=50):
    for nombre in doc._.personajes:
        conteo[nombre] += 1

# conteo.most_common(): entrega los pares (nombre, cantidad) de mayor a menor
pares_de_mayor_a_menor = conteo.most_common()
# columns=: los nombres de las dos columnas de la tabla
apariciones_por_personaje = pd.DataFrame(pares_de_mayor_a_menor, columns=["personaje", "apariciones"])
apariciones_por_personaje

## 6. Guardar la herramienta

Todo lo anterior vive en memoria y se pierde al cerrar el cuaderno. `to_disk()`
lo guarda entero: modelo, reglas, componentes y configuración.

Pero hay una trampa, y es la razón por la que arriba les puse `_demo` a los
nombres: **spaCy guarda QUÉ componentes tiene el pipeline, pero no guarda su
código.** Al volver a cargarlo, ese código tiene que estar disponible en la
sesión. Y si vive en una celda de un cuaderno, hay que acordarse de correr esa
celda.

Por eso los componentes definitivos están en `componentes.py`, al lado de este
cuaderno. **Abrilo:** son las mismas dos funciones que acabás de escribir, con
un par de detalles más.

In [ ]:
# import componentes: lee el archivo componentes.py y registra sus funciones en spaCy
import componentes

# Verificamos que el diccionario del archivo se haya cargado
cantidad_de_correcciones = len(componentes.CORRECCIONES_VOSEO)
print("Correcciones de voseo cargadas:", cantidad_de_correcciones)

Ahora armamos el pipeline definitivo desde cero, con los componentes que vienen
del archivo.

In [ ]:
# Cargamos el modelo base para armar la versión final
nlp_final = spacy.load("es_core_news_lg")

# 1. Agregamos el entity_ruler antes del ner (igual que antes)
configuracion_de_reglas = {"overwrite_ents": True}
reglas_finales = nlp_final.add_pipe("entity_ruler", config=configuracion_de_reglas, before="ner")
reglas_finales.add_patterns(patrones)

# 2. Agregamos los componentes usando los nombres registrados en componentes.py
nlp_final.add_pipe("corrector_voseo", last=True)
nlp_final.add_pipe("detector_personajes", last=True)

print("Pipeline definitivo listo:")
print(nlp_final.pipe_names)

In [ ]:
# os: para mirar los archivos de una carpeta.
import os

# to_disk(): guarda el modelo completo, reglas y configuración en una carpeta física
nlp_final.to_disk("pipeline_propio")

print("Archivos generados dentro de pipeline_propio/:")
# os.listdir: lista los archivos de la carpeta. sorted(): los ordena alfabéticamente
archivos_guardados = os.listdir("pipeline_propio")
archivos_ordenados = sorted(archivos_guardados)
for archivo in archivos_ordenados:
    print(" •", archivo)

Y ahora se recarga sin problemas, en esta sesión o en cualquier otra, siempre
que antes se haga `import componentes`.

In [ ]:
# spacy.load también puede cargar una carpeta guardada con to_disk
nlp_recargado = spacy.load("pipeline_propio")
print("Pipeline recargado con éxito:", nlp_recargado.pipe_names)

# Probamos que tanto el voseo como los personajes y entidades funcionen juntos
doc = nlp_recargado("Vos sabés que Mirta Bertotti vivía en Mercedes.")

print()
print("1. Verbos corregidos (lema):")
# pos_: Part of Speech como texto legible ("VERB" = verbo)
for token in doc:
    if token.pos_ == "VERB":
        print(f" • {token.text} -> {token.lemma_}")

personajes_detectados = doc._.personajes
print()
print(f"2. Atributo propio: {personajes_detectados}")

print()
print("3. Entidades finales:")
for ent in doc.ents:
    print(f" • {ent.text} -> {ent.label_}")

## 7. Ejercicio

1. Agregá tres entradas nuevas a `datos/personajes.txt` —mirá el corpus para
   elegirlas— y volvé a correr desde la celda del `PhraseMatcher`. **Anotá en tu
   bitácora por qué elegiste esas tres.**

2. Ampliá `CORRECCIONES_VOSEO` con cinco formas más. Fijate cuántas apariciones
   nuevas captura sobre 800 párrafos.

3. Escribí un componente propio que marque los párrafos que contienen diálogo
   (los que tienen `LUCAS.-` o `ALEX.-`) con una extensión `doc._.es_dialogo`.
   ¿Qué proporción del corpus es diálogo?

In [ ]:
# --- EJERCICIO 3: Detector de diálogo ---

# Paso 1: Creamos la extensión booleana (por defecto False)
Doc.set_extension("es_dialogo", default=False, force=True)

# Paso 2: Definimos el componente
@Language.component("detector_dialogo")
def detector_dialogo(doc):
    """Marca con True si el párrafo contiene marcas de diálogo (LUCAS.- o ALEX.-)."""
    # Arrancamos suponiendo que no hay diálogo, y lo cambiamos si aparece alguna marca.
    # doc.text: texto completo del párrafo analizado
    hay_dialogo = False
    if "LUCAS.-" in doc.text:
        hay_dialogo = True
    if "ALEX.-" in doc.text:
        hay_dialogo = True
    doc._.es_dialogo = hay_dialogo
    return doc

# Paso 3: Agregamos el componente al pipeline
nlp_recargado.add_pipe("detector_dialogo", last=True)

# Paso 4: Calculamos la proporción sobre 300 párrafos
primeros = corpus.head(300)
muestra = list(primeros["texto"])
total_dialogos = 0

for doc in nlp_recargado.pipe(muestra):
    if doc._.es_dialogo:
        total_dialogos += 1

cantidad_de_la_muestra = len(muestra)
porcentaje = (total_dialogos / cantidad_de_la_muestra) * 100
# :.1f: con un decimal
print(f"Párrafos con diálogo: {total_dialogos} de {cantidad_de_la_muestra} ({porcentaje:.1f}%)")

> ### Perilla
>
> El `EntityRuler` lo pusimos **antes** del `ner` y con `overwrite_ents=True`.
> Las dos cosas se pueden cambiar, y cambian el resultado.
>
> Corré la celda de abajo: arma el mismo pipeline con el ruler **después** del
> `ner`. Mirá qué pasa con las entidades que se pisan.
>
> Cuando dos componentes opinan distinto sobre el mismo pedazo de texto, alguien
> tiene que ganar. Vos elegís quién.

In [ ]:
# Cargamos otro modelo para ver qué pasa cuando cambiamos el orden
nlp_al_reves = spacy.load("es_core_news_lg")

# after="ner": colocamos el EntityRuler DESPUÉS del reconocedor estadístico
# (Al no poner overwrite_ents=True, gana lo que ya decidió el modelo ner)
reglas_despues = nlp_al_reves.add_pipe("entity_ruler", after="ner")
reglas_despues.add_patterns(patrones)

frase = "Mirta Bertotti vivía en Mercedes y Chiri fundó la Revista Orsai en Barcelona."

doc_con_overwrite = nlp_mejorado(frase)
doc_sin_overwrite = nlp_al_reves(frase)

print("1. Reglas ANTES del ner (nuestras reglas ganan):")
for ent in doc_con_overwrite.ents:
    print(f" • {ent.text} -> {ent.label_}")

print()
print("2. Reglas DESPUÉS del ner (el modelo estadístico gana):")
for ent in doc_sin_overwrite.ents:
    print(f" • {ent.text} -> {ent.label_}")

## Lo que hiciste hoy

- Usaste `Matcher` para describir estructuras gramaticales y `PhraseMatcher` para
  listas de términos.
- Metiste tus reglas adentro del pipeline con `EntityRuler` y decidiste quién
  gana cuando hay conflicto.
- Escribiste dos componentes propios y aprendiste que el orden en el pipeline
  cambia el resultado.
- Le agregaste al `Doc` un atributo que spaCy no trae.
- Guardaste la herramienta configurada en `pipeline_propio/`, con los
  componentes en un archivo aparte para que se pueda recargar.

## Lo que viene

Tenés la herramienta lista, pero hasta ahora la corriste sobre muestras de unos
cientos de párrafos. En el `06` la soltamos sobre el corpus completo, y ahí
aparecen problemas nuevos.